# Attention-PestNet Baseline Model Training (HSDPA, r=1.00) - Stage 1
**Thesis Topic:** A Modified Attention-PestNet with Channel-Reduced Hierarchical Scaled Dot-Product Attention for Efficient Insect Pest Detection  
**Authors:** Lean Adrian Murillo, James Oliver C. Mendoza, DM Rashid P. Ferrer, Charisse P. Barbosa (University of Mindanao)  
**Target Architecture:** Original Baseline Attention-PestNet (`cfg/hsdpa.yaml`, r=1.00, unreduced HSDPA)  
**Dataset:** IP102-YOLO (`leanadrianmurillo/ip102-yolo`)  
**Training Stage:** Stage 1 (Epochs 1 – 50 from scratch)  
---
This notebook sets up the environment, verifies the official IP102-YOLO dataset, trains the baseline Attention-PestNet architecture from scratch on Kaggle Dual NVIDIA T4 GPUs, and executes the Section 2.7 Thesis Evaluation & Efficiency Benchmark.

In [ ]:
# 1. Setup repository and install ultralytics in editable mode
import os
os.environ["WANDB_MODE"] = "disabled"
import sys
import shutil

REPO_DIR = "/kaggle/working/HSDPA"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/90377Sednaaa/HSDPA.git {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull

%cd {REPO_DIR}
!pip install -q -e .

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print("Environment successfully set up at:", REPO_DIR)


In [ ]:
# 2. Check GPU environment and CUDA capabilities
import torch
from ultralytics import YOLO

print("PyTorch version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
device_count = torch.cuda.device_count()
print("GPU Count:", device_count)
for i in range(device_count):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")


In [ ]:
# 3. Verify IP102-YOLO dataset directory and images
import os, glob, yaml

DATA_ROOT = "/kaggle/input/ip102-yolo"
if not os.path.exists(DATA_ROOT):
    for root, dirs, files in os.walk("/kaggle/input"):
        if "images" in dirs and os.path.exists(os.path.join(root, "images/train")):
            DATA_ROOT = root
            break

print(f"Dataset root verified at: {DATA_ROOT}")
train_count = len(os.listdir(os.path.join(DATA_ROOT, "images/train")))
val_count = len(os.listdir(os.path.join(DATA_ROOT, "images/val")))
test_count = len(os.listdir(os.path.join(DATA_ROOT, "images/test"))) if os.path.exists(os.path.join(DATA_ROOT, "images/test")) else 0
print(f"Train images: {train_count} (Expected: 15,178)")
print(f"Validation images: {val_count} (Expected: 3,798)")
print(f"Test images: {test_count} (Expected: 3,798)")

# Update path in cfg/ip102.yaml dynamically
with open("cfg/ip102.yaml", "r") as f:
    ip102_cfg = yaml.safe_load(f)
ip102_cfg["path"] = DATA_ROOT
with open("cfg/ip102.yaml", "w") as f:
    yaml.dump(ip102_cfg, f, default_flow_style=False, sort_keys=False)
print("cfg/ip102.yaml dynamically configured and ready!")


In [ ]:
# 4. Train Baseline Attention-PestNet Model (HSDPA, r=1.00 - Stage 1)
import torch
import os, glob
os.environ["WANDB_MODE"] = "disabled"
from ultralytics import YOLO

MODEL_CFG = "cfg/hsdpa.yaml"
PROJECT_NAME = "dataset_r100_stage1"
EXP_NAME = "ip102_baseline"

# Seamless resume check if previous run interrupted
resume_ckpt = None
candidate_ckpts = glob.glob(f"/kaggle/input/**/{EXP_NAME}/weights/last.pt", recursive=True) + \
                  glob.glob(f"{PROJECT_NAME}/{EXP_NAME}/weights/last.pt")

if candidate_ckpts:
    resume_ckpt = candidate_ckpts[0]
    print(f"Found existing checkpoint to resume: {resume_ckpt}")
    model = YOLO(resume_ckpt)
    resume_flag = True
else:
    print(f"Initializing fresh baseline Attention-PestNet model from {MODEL_CFG}")
    model = YOLO(MODEL_CFG)
    resume_flag = False

# Multi-GPU support for dual T4
device_arg = [0, 1] if torch.cuda.device_count() >= 2 else 0

# Train for Stage 1 (50 epochs) matching baseline protocol
results = model.train(
    data="cfg/ip102.yaml",
    epochs=50,                # Stage 1 target (Epochs 1-50)
    batch=32,                 # Effective batch size matching baseline study
    imgsz=640,
    device=device_arg,
    optimizer="SGD",          # SGD with momentum
    lr0=0.01,
    momentum=0.9,
    amp=True,
    save_period=10,           # Save checkpoints every 10 epochs
    project=PROJECT_NAME,
    name=EXP_NAME,
    exist_ok=True,
    workers=4,
    resume=resume_flag,
    verbose=True
)


In [ ]:
# 5. Comprehensive Thesis Evaluation & Efficiency Benchmark (Section 2.7 Protocol)
import time, os, glob
import torch
import pandas as pd
from ultralytics import YOLO

best_ckpt = f"{PROJECT_NAME}/{EXP_NAME}/weights/best.pt"
if os.path.exists(best_ckpt):
    print(f"=== Evaluating Baseline Best Checkpoint: {best_ckpt} ===")
    eval_model = YOLO(best_ckpt)
    
    # 1. Detection Performance (mAP@50, mAP@50:95, Precision, Recall)
    val_results = eval_model.val(data="cfg/ip102.yaml", split="val", batch=32, imgsz=640, device=0)
    p = val_results.box.mp
    r = val_results.box.mr
    map50 = val_results.box.map50
    map50_95 = val_results.box.map
    
    # 2. Computational Efficiency Benchmark (Section 2.7: FP32, batch 1, 20 warmup, 200 timed runs)
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    bench_model = eval_model.model.to(device).float()
    bench_model.eval()
    dummy_input = torch.randn(1, 3, 640, 640, device=device)
    
    with torch.no_grad():
        for _ in range(20):
            _ = bench_model(dummy_input)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        
        start_time = time.perf_counter()
        for _ in range(200):
            _ = bench_model(dummy_input)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
        total_time = time.perf_counter() - start_time
    
    latency_ms = (total_time / 200.0) * 1000.0
    fps = 200.0 / total_time
    total_params = sum(x.numel() for x in eval_model.model.parameters())
    model_size_mb = os.path.getsize(best_ckpt) / (1024 * 1024)
    
    print("\n" + "=" * 65)
    print("   THESIS EVALUATION METRICS SUMMARY - BASELINE STAGE 1 (Section 2.7)")
    print("=" * 65)
    print(f"  Precision (P):          {p:.4f} ({p*100:.2f}%)")
    print(f"  Recall (R):             {r:.4f} ({r*100:.2f}%)")
    print(f"  mAP@50:                 {map50:.4f} ({map50*100:.2f}%)")
    print(f"  mAP@50:95:              {map50_95:.4f} ({map50_95*100:.2f}%)")
    print(f"  Parameters:             {total_params:,}")
    print(f"  Inference Latency:      {latency_ms:.2f} ms")
    print(f"  Throughput (FPS):       {fps:.2f} FPS")
    print(f"  Model File Size:        {model_size_mb:.2f} MB")
    print("=" * 65)
    
    summary_df = pd.DataFrame([{
        "Model": "Attention-PestNet Baseline (Stage 1)",
        "Precision": p,
        "Recall": r,
        "mAP@50": map50,
        "mAP@50-95": map50_95,
        "Parameters": total_params,
        "Latency_ms": latency_ms,
        "FPS": fps,
        "ModelSize_MB": model_size_mb
    }])
    out_csv = f"{PROJECT_NAME}/{EXP_NAME}/thesis_metrics_summary_baseline_stage1.csv"
    summary_df.to_csv(out_csv, index=False)
    print(f"Saved Baseline Stage 1 metrics summary to: {out_csv}")
else:
    print(f"Checkpoint {best_ckpt} not found.")
